In [1]:
!pip install --upgrade transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 104.5 MB/s eta 0:00:00


In [2]:
"""Scoring logic for the structured output task.

Provides parsers for each format, field matching, and scoring functions.
"""

import csv
import io
import json
import xml.etree.ElementTree as ET
from collections import defaultdict


# ---------------------------------------------------------------------------
# Format parsers — each returns a dict or None on failure
# ---------------------------------------------------------------------------

def parse_json(text: str) -> dict | None:
    """Try to parse text as JSON. Return dict or None."""
    try:
        result = json.loads(text.strip())
        if isinstance(result, dict):
            return result
        return None
    except (json.JSONDecodeError, ValueError):
        return None


def parse_yaml(text: str) -> dict | None:
    """Try to parse text as YAML. Return dict or None."""
    try:
        import yaml
        result = yaml.safe_load(text.strip())
        if isinstance(result, dict):
            return result
        return None
    except Exception:
        return None


def parse_xml(text: str) -> dict | None:
    """Parse XML, extract child elements of <record> as dict."""
    try:
        text = text.strip()
        root = ET.fromstring(text)
        result = {}
        for child in root:
            result[child.tag] = child.text if child.text else ""
        if result:
            return result
        return None
    except ET.ParseError:
        return None


def parse_csv(text: str) -> dict | None:
    """Parse CSV (header + data row) into dict."""
    try:
        text = text.strip()
        reader = csv.DictReader(io.StringIO(text))
        rows = list(reader)
        if len(rows) >= 1:
            return dict(rows[0])
        return None
    except Exception:
        return None


def parse_toml(text: str) -> dict | None:
    """Try to parse text as TOML. Return dict or None."""
    try:
        # Python 3.11+ has tomllib in stdlib
        try:
            import tomllib
        except ImportError:
            import tomli as tomllib
        result = tomllib.loads(text.strip())
        if isinstance(result, dict):
            return result
        return None
    except Exception:
        return None


PARSERS = {
    "json": parse_json,
    "yaml": parse_yaml,
    "xml": parse_xml,
    "csv": parse_csv,
    "toml": parse_toml,
}


# ---------------------------------------------------------------------------
# Field matching
# ---------------------------------------------------------------------------

def _normalize_value(val) -> str:
    """Normalize a value to string for comparison."""
    if val is None:
        return ""
    return str(val).strip().lower()


def _values_match(predicted, expected) -> bool:
    """Check if predicted value matches expected value."""
    # Try numeric comparison first
    try:
        pred_num = float(str(predicted))
        exp_num = float(str(expected))
        # For integers, exact match
        if isinstance(expected, int):
            return int(pred_num) == exp_num
        # For floats, tolerance of 0.01
        return abs(pred_num - exp_num) < 0.01
    except (ValueError, TypeError):
        pass

    # Fall back to case-insensitive string comparison
    return _normalize_value(predicted) == _normalize_value(expected)


def match_fields(predicted: dict | None, ground_truth: dict) -> float:
    """Return fraction of ground truth fields correctly matched in predicted.

    Args:
        predicted: Parsed output dict, or None if parsing failed.
        ground_truth: Expected field values.

    Returns:
        Float between 0.0 and 1.0.
    """
    if predicted is None or not ground_truth:
        return 0.0

    correct = 0
    total = len(ground_truth)

    for key, expected_val in ground_truth.items():
        if key in predicted and _values_match(predicted[key], expected_val):
            correct += 1

    return correct / total


# ---------------------------------------------------------------------------
# Per-sample and aggregate scoring
# ---------------------------------------------------------------------------

def score_sample(predicted_text: str, ground_truth: dict, format_name: str) -> dict:
    """Score a single prediction.

    Args:
        predicted_text: The model's raw output string.
        ground_truth: Dict of expected field values.
        format_name: The target format ("json", "yaml", etc.).

    Returns:
        Dict with "valid" (0/1), "field_accuracy" (float), "score" (float).
    """
    parser = PARSERS.get(format_name)
    if parser is None:
        return {"valid": 0, "field_accuracy": 0.0, "score": 0.0}

    parsed = parser(predicted_text)
    valid = 1 if parsed is not None else 0
    field_accuracy = match_fields(parsed, ground_truth)
    score = 0.5 * valid + 0.5 * field_accuracy

    return {
        "valid": valid,
        "field_accuracy": field_accuracy,
        "score": score,
    }


def score_all(predictions: list[dict], ground_truths: list[dict]) -> dict:
    """Score all predictions against ground truths.

    Args:
        predictions: List of dicts with "prediction" and "format" keys.
        ground_truths: List of dicts with "fields" and "format" keys.

    Returns:
        Dict with "overall" score, "per_format" breakdown, and "details".
    """
    assert len(predictions) == len(ground_truths), (
        f"Prediction count ({len(predictions)}) != ground truth count ({len(ground_truths)})"
    )

    details = []
    format_scores = defaultdict(list)

    for pred, gt in zip(predictions, ground_truths):
        fmt = gt["format"]
        result = score_sample(pred["prediction"], gt["fields"], fmt)
        result["format"] = fmt
        details.append(result)
        format_scores[fmt].append(result["score"])

    per_format = {}
    for fmt, scores in sorted(format_scores.items()):
        per_format[fmt] = round(sum(scores) / len(scores), 4)

    all_scores = [d["score"] for d in details]
    overall = round(sum(all_scores) / len(all_scores), 4) if all_scores else 0.0
    valid_count = sum(d["valid"] for d in details)

    return {
        "overall": overall,
        "per_format": per_format,
        "total_samples": len(details),
        "valid_count": valid_count,
        "valid_ratio": round(valid_count / len(details), 4) if details else 0.0,
        "details": details,
    }

In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import json
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

from transformers import StoppingCriteria, StoppingCriteriaList


SYSTEM_PROMPT = (
    "You convert natural language into structured data.\n"
    "STRICT RULES:\n"
    "- Output ONLY valid {format}\n"
    "- No explanations\n"
    "- No extra text\n"
)

STOP_STRINGS = {
    "json": ["}", "<|user|>", "<|assistant|>"],
    "xml": ["</record>", "<|user|>", "<|assistant|>"],
    "yaml": ["\n<|", "<|user|>", "<|assistant|>"],
    "csv": ["\n<|", "<|user|>", "<|assistant|>"],
    "toml": ["\n<|", "<|user|>", "<|assistant|>"],
}


class StopOnTokens(StoppingCriteria):
    def __init__(self, stop_token_ids):
        self.stop_token_ids = stop_token_ids  # list of lists

    def __call__(self, input_ids, scores, **kwargs):
        for stop_ids in self.stop_token_ids:
            if len(input_ids[0]) >= len(stop_ids):
                if input_ids[0][-len(stop_ids):].tolist() == stop_ids:
                    return True
        return False


def build_stopping_criteria(tokenizer, format_name):
    stop_strings = STOP_STRINGS.get(format_name, [])

    stop_token_ids = []
    for s in stop_strings:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) > 0:
            stop_token_ids.append(ids)

    if not stop_token_ids:
        return None

    return StoppingCriteriaList([StopOnTokens(stop_token_ids)])

def apply_stop(text, format_name):
    stops = STOP_STRINGS.get(format_name, [])
    for s in stops:
        idx = text.find(s)
        if idx != -1:
            return text[: idx + len(s)]
    return text


def generate(model, tokenizer, prompt: str, format_name: str) -> str:
    system_prompt = SYSTEM_PROMPT.format(format=format_name.upper())

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    stopping_criteria = build_stopping_criteria(tokenizer, format_name)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,  # 🔥 ADD THIS
            use_cache=True,
            stopping_criteria=stopping_criteria,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)
    response = apply_stop(response, format_name)

    for bad in ["user", "assistant"]:
        idx = response.find(bad)
        if idx != -1:
            response = response[:idx]

    # optional safety trim (keep it as backup)
    for s in STOP_STRINGS.get(format_name, []):
        idx = response.find(s)
        if idx != -1:
            response = response[: idx + len(s)]
            break

    del inputs, output_ids, generated
    torch.cuda.empty_cache()

    return response.strip()

In [4]:
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
# )

MODEL_NAME = "Qwen/Qwen3.5-0.8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    # quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

adapter_path = "/kaggle/input/datasets/abukanabek/hometask-2026-submission"
model = PeftModel.from_pretrained(model, adapter_path)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [5]:
test_path = "/kaggle/input/datasets/abukanabek/hometask-2026-data/valid.jsonl"
output_path = "result.jsonl"

lmt = -1

# Load test data
print(f"Loading test data from: {test_path}")  # This will be later substituted for our test sample
test_samples = []
with open(test_path, encoding="utf-8") as f:
    it = 0
    for line in f:
        test_samples.append(json.loads(line))
        it += 1
        if lmt != -1 and lmt <= it:
            break

# Run inference
print(f"Running inference on {len(test_samples)} samples...")
predictions = []
for i, sample in tqdm(enumerate(test_samples), total=len(test_samples)):
    pred = generate(model, tokenizer, sample["input"], sample["format"])
    predictions.append({
        "input": sample["input"],
        "format": sample["format"],
        "prediction": pred,
    })
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(test_samples)} done")

# Load ground truth
ground_truths = []
with open(test_path, encoding="utf-8") as f:
    it = 0
    for line in f:
        ground_truths.append(json.loads(line))
        it += 1
        if lmt != -1 and lmt <= it:
            break

# Score
results = score_all(predictions, ground_truths)

# Print summary
print("\n" + "=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"Overall score: {results['overall']:.4f}")
print(f"Valid outputs: {results['valid_count']}/{results['total_samples']} "
      f"({results['valid_ratio']:.1%})")
print("\nPer-format scores:")
for fmt, score in sorted(results["per_format"].items()):
    print(f"  {fmt:>5s}: {score:.4f}")
print("=" * 50)

# Save detailed results
Path(output_path).parent.mkdir(parents=True, exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"\nDetailed results saved to: {output_path}")

Loading test data from: /kaggle/input/datasets/abukanabek/hometask-2026-data/valid.jsonl
Running inference on 1000 samples...


  0%|          | 0/1000 [00:00<?, ?it/s]

  50/1000 done
  100/1000 done
  150/1000 done
  200/1000 done
  250/1000 done
  300/1000 done
  350/1000 done
  400/1000 done
  450/1000 done
  500/1000 done
  550/1000 done
  600/1000 done
  650/1000 done
  700/1000 done
  750/1000 done
  800/1000 done
  850/1000 done
  900/1000 done
  950/1000 done
  1000/1000 done

EVALUATION RESULTS
Overall score: 1.0000
Valid outputs: 1000/1000 (100.0%)

Per-format scores:
    csv: 1.0000
   json: 1.0000
   toml: 1.0000
    xml: 1.0000
   yaml: 1.0000

Detailed results saved to: result.jsonl


In [6]:
predicitions_output_path = "predictions.jsonl"

Path(predicitions_output_path).parent.mkdir(parents=True, exist_ok=True)
with open(predicitions_output_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2, ensure_ascii=False)
print(f"\nDetailed predictions saved to: {predicitions_output_path}")


Detailed predictions saved to: predictions.jsonl
